# Stage A / NB 02 — Fold leakage audit, fold regeneration, and Table 1

Protocol reference: Section 3.1 (mandatory pre-flight), Section 4 (split protocol),
Section 9 Stage A NB 02. Addresses referee 2b and the AE's statistics request at its root:
no statistical treatment can rescue a leaky split.

## Why this notebook exists

A local audit of the inherited folds in `multi_task_CV/` found study-level contamination:

| check | result |
| --- | --- |
| study directories spanning two or more *test* folds | 409 pairwise occurrences |
| study directories present in BOTH train and test of the same fold | 135-153 per fold |

The `fold` column in `covid_midrc_dataset.csv` is assigned per image, not per study, and
386 study directories hold 2 images while 55 hold 3 or more. Two frontal views of one
acquisition therefore land on opposite sides of the fold boundary. The published
out-of-fold numbers (mRALE MAE 3.88, COVID balanced accuracy 0.616) are optimistically
biased by an unknown amount.

This notebook does three things, in order:

1. **Audit** the inherited folds and publish the evidence (`fold_leakage_audit.csv`).
2. **Regenerate** folds with `StratifiedGroupKFold` on `group_id`, stratified by
   PCR status crossed with mRALE severity band.
3. **Rebuild** the multitask Harmony JSONL files on the new folds, reusing the exact prompt
   templates harvested from the legacy files so that Stage B trains on identical text.

Per the protocol: folds are regenerated, not patched, and every downstream result is
recomputed from the new folds.

Outputs (under `stage_A/nb02_folds/`)
- `fold_leakage_audit.csv`, `legacy_fold_audit_summary.json`
- `fold_definitions/midrc_folds_v2.csv`
- `fold_definitions/multitask_{train,test}_fold_{k}_harmony.jsonl`
- `prompt_templates.json`
- `cohort_composition_table1.csv`  (-> manuscript Table 1)
- `new_fold_audit.csv`, `gate_nb02.json`

## 1. Imports and configuration

In [ ]:
import csv
import json
import math
import os
import random
import sys
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N_FOLDS = 5
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_PROJECT_ROOT = Path("/data/liangz2/openi/midrc")
FALLBACK_STAGE_A_DIR = FALLBACK_PROJECT_ROOT / "tetci_resubmit" / "stage_A"
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
]

stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Loaded path contract from:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
for directory in [NB02_DIR, FOLD_DEF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LEGACY_CV_DIR = Path(stage_paths["legacy"]["multitask_cv_dir"])
LEGACY_TRAIN_PATTERN = "multitask_train_fold_{fold}_harmony.jsonl"
LEGACY_TEST_PATTERN = "multitask_test_fold_{fold}_harmony.jsonl"

HARMONY_TRUTHFULNESS_JSONL = stage_paths["datasets"].get("combined_cxr_harmony_train_jsonl")
IMAGE_PATH_REWRITES = OrderedDict(stage_paths["image_path_rewrites"])

MANIFEST_CSV = NB01_DIR / "midrc_manifest.csv"
if not MANIFEST_CSV.is_file():
    raise FileNotFoundError(f"{MANIFEST_CSV} not found. Run NB 01 first.")

# --- Protocol decisions, all explicit and all recorded ----------------------------------
# Montgomery is held out entirely as external cohort X1 (protocol Section 3.3): every
# Montgomery case is COVID-negative, so including them in training creates a source-label
# confound. The legacy folds put 110 Montgomery normality records into each training file.
# Consequence: the normality_classification task has no MIDRC training data and is DROPPED
# from the regenerated multitask folds. Stage B therefore trains two direct tasks
# (mrale_prediction, covid_classification) plus their truthfulness auxiliaries.
INCLUDE_MONTGOMERY_IN_TRAINING = False
INCLUDE_NORMALITY_TASK = False

# Auxiliary truthfulness records are reused verbatim from combined_cxr_harmony_train.jsonl
# and re-assigned to the new folds by file_name, so no prompt text is invented here.
INCLUDE_TRUTHFULNESS_TASKS = True

REBUILD_HARMONY_JSONL = True

print("Manifest:", MANIFEST_CSV)
print("Legacy CV dir:", LEGACY_CV_DIR, "exists:", LEGACY_CV_DIR.is_dir())
print("Truthfulness source:", HARMONY_TRUTHFULNESS_JSONL)
print("Output:", NB02_DIR)

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_number}: {exc}") from exc
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return path


def rewrite_image_path(path):
    text = str(path)
    for old_prefix, new_prefix in IMAGE_PATH_REWRITES.items():
        if text.startswith(old_prefix):
            return new_prefix + text[len(old_prefix):]
    return text


def study_dir_of(image_path):
    return str(Path(rewrite_image_path(image_path)).parent)


manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest rows: {len(manifest):,}")
primary = manifest[manifest["in_primary_cohort"] == True].reset_index(drop=True)
print(f"Primary cohort: {len(primary):,} images, "
      f"{primary['group_id'].nunique():,} grouping units")

## 2. Audit the inherited folds

Three checks per fold, and one across folds. Every finding becomes a row in
`fold_leakage_audit.csv` so that the evidence is citable in the response letter rather than
asserted.

In [ ]:
audit_rows = []
legacy_available = LEGACY_CV_DIR.is_dir() and all(
    (LEGACY_CV_DIR / pattern.format(fold=fold)).is_file()
    for fold in range(N_FOLDS)
    for pattern in [LEGACY_TRAIN_PATTERN, LEGACY_TEST_PATTERN]
)

legacy_summary = {"available": legacy_available}

if not legacy_available:
    print("Legacy fold files not all present; skipping the legacy audit.")
    print("This is acceptable only if you have never trained on them. Otherwise copy "
          f"{LEGACY_CV_DIR} to this node so the audit can be published.")
else:
    legacy = {}
    for fold in range(N_FOLDS):
        train = read_jsonl(LEGACY_CV_DIR / LEGACY_TRAIN_PATTERN.format(fold=fold))
        test = read_jsonl(LEGACY_CV_DIR / LEGACY_TEST_PATTERN.format(fold=fold))
        legacy[fold] = {"train": train, "test": test}

    def image_and_group_sets(records, midrc_only=True):
        images, groups = set(), set()
        for record in records:
            path = rewrite_image_path(record["image_path"])
            if midrc_only and "Montgomery" in path:
                continue
            images.add(Path(path).name)
            groups.add(str(Path(path).parent))
        return images, groups

    per_fold = {}
    for fold in range(N_FOLDS):
        train_images, train_groups = image_and_group_sets(legacy[fold]["train"])
        test_images, test_groups = image_and_group_sets(legacy[fold]["test"])
        per_fold[fold] = {
            "train_images": train_images, "train_groups": train_groups,
            "test_images": test_images, "test_groups": test_groups,
        }
        image_overlap = train_images & test_images
        group_overlap = train_groups & test_groups
        audit_rows.append({
            "scope": f"fold_{fold}",
            "check": "train_test_image_overlap",
            "value": len(image_overlap),
            "status": "PASS" if not image_overlap else "FAIL",
            "detail": "; ".join(sorted(image_overlap)[:5]),
        })
        audit_rows.append({
            "scope": f"fold_{fold}",
            "check": "train_test_study_group_overlap",
            "value": len(group_overlap),
            "status": "PASS" if not group_overlap else "FAIL",
            "detail": "; ".join(Path(item).name for item in sorted(group_overlap)[:3]),
        })
        print(f"fold {fold}: train images={len(train_images):,} test images={len(test_images):,} "
              f"| image overlap={len(image_overlap)} | STUDY-GROUP overlap={len(group_overlap)}")

    print()
    total_test_group_overlap = 0
    for left in range(N_FOLDS):
        for right in range(left + 1, N_FOLDS):
            overlap = per_fold[left]["test_groups"] & per_fold[right]["test_groups"]
            total_test_group_overlap += len(overlap)
            audit_rows.append({
                "scope": f"test_fold_{left}_vs_{right}",
                "check": "cross_test_fold_study_group_overlap",
                "value": len(overlap),
                "status": "PASS" if not overlap else "FAIL",
                "detail": "; ".join(Path(item).name for item in sorted(overlap)[:3]),
            })
            if overlap:
                print(f"  test folds {left} & {right}: {len(overlap)} shared study groups")

    montgomery_in_train = sum(
        1 for fold in range(N_FOLDS)
        for record in legacy[fold]["train"]
        if "Montgomery" in rewrite_image_path(record["image_path"])
    )
    audit_rows.append({
        "scope": "all_folds", "check": "montgomery_records_in_training",
        "value": montgomery_in_train,
        "status": "PASS" if montgomery_in_train == 0 else "FAIL",
        "detail": "Protocol Section 3.3 holds Montgomery out entirely as cohort X1.",
    })

    legacy_summary.update({
        "total_cross_test_fold_group_overlaps": total_test_group_overlap,
        "train_test_group_overlap_per_fold": {
            str(fold): len(per_fold[fold]["train_groups"] & per_fold[fold]["test_groups"])
            for fold in range(N_FOLDS)
        },
        "montgomery_records_in_training": montgomery_in_train,
    })

    print()
    print(f"TOTAL cross-test-fold study-group overlaps: {total_test_group_overlap}")
    print(f"Montgomery records inside training files: {montgomery_in_train}")

In [ ]:
# The fold column of the source CSV is audited directly, which identifies the root cause.
csv_group_folds = primary.groupby("group_id")["legacy_fold"].nunique()
csv_split_groups = int((csv_group_folds > 1).sum())
audit_rows.append({
    "scope": "covid_midrc_dataset.csv",
    "check": "group_spans_multiple_legacy_folds",
    "value": csv_split_groups,
    "status": "PASS" if csv_split_groups == 0 else "FAIL",
    "detail": "The fold column is assigned per image rather than per study group.",
})

audit_frame = pd.DataFrame(audit_rows, columns=["scope", "check", "value", "status", "detail"])
audit_frame.to_csv(NB02_DIR / "fold_leakage_audit.csv", index=False)

legacy_summary["csv_groups_spanning_multiple_folds"] = csv_split_groups
legacy_summary["audit_failures"] = int((audit_frame["status"] == "FAIL").sum())
legacy_summary["audited_utc"] = datetime.now(timezone.utc).isoformat()
with (NB02_DIR / "legacy_fold_audit_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(legacy_summary, handle, indent=2)

print(f"Study groups spanning more than one legacy fold (source CSV): {csv_split_groups}")
print()
print(audit_frame.groupby(["check", "status"]).size().to_string())
print()
LEGACY_FOLDS_ARE_CLEAN = int((audit_frame["status"] == "FAIL").sum()) == 0
if LEGACY_FOLDS_ARE_CLEAN:
    print("Legacy folds passed every check. Regeneration below is a no-op reproduction.")
else:
    print("Legacy folds FAILED the audit. Regenerating folds in Section 3.")
    print("Every Stage B, C, and D result must be recomputed on the new folds. Do not mix "
          "old and new numbers in a single table.")

## 3. Regenerate folds: StratifiedGroupKFold on `group_id`

Grouping unit: `group_id` from NB 01 (study directory, with cross-study near-duplicate pairs
merged). Stratum: PCR status crossed with mRALE severity band, taken from the *group* rather
than the image, so a group belongs to exactly one stratum.

`sklearn.model_selection.StratifiedGroupKFold` is used when available. A deterministic greedy
fallback is included because it is a scikit-learn 1.0+ API and Biowulf module versions vary:
strata are filled largest-group-first into whichever fold is currently most deficient in that
stratum, which is the standard approximation and is exactly reproducible under a fixed seed.

In [ ]:
# A grouping unit needs ONE label to be stratified on. Where NB 01's near-duplicate merging
# produced a mixed-label group, the stratum label is decided by majority vote with a
# deterministic tie-break (more positives wins; on an exact tie, "Yes" wins because it is the
# majority class and keeps the scarce negatives spread across folds).
#
# This affects FOLD BALANCING ONLY. Every image keeps its own PCR label, and every metric in
# the paper is computed from per-image labels, so no ground truth is altered here.
def majority_covid_label(values):
    counts = Counter(values)
    positives, negatives = counts.get("Yes", 0), counts.get("No", 0)
    if positives != negatives:
        return "Yes" if positives > negatives else "No"
    return "Yes"


group_frame = (
    primary.groupby("group_id")
    .agg(
        n_images=("filename", "size"),
        covid_positive=("covid_positive", majority_covid_label),
        n_covid_labels=("covid_positive", "nunique"),
        n_covid_positive=("covid_positive", lambda values: int((values == "Yes").sum())),
        n_covid_negative=("covid_positive", lambda values: int((values == "No").sum())),
        mrale_mean=("mrale_total_annotated", "mean"),
        mrale_max=("mrale_total_annotated", "max"),
    )
    .reset_index()
)

mixed_label_groups = group_frame[group_frame["n_covid_labels"] > 1]
print(f"Grouping units with mixed PCR labels: {len(mixed_label_groups)}")
if len(mixed_label_groups):
    print("  Stratum label assigned by majority vote (fold balancing only; per-image labels "
          "and all metrics are unchanged):")
    print(mixed_label_groups[[
        "group_id", "n_images", "n_covid_positive", "n_covid_negative", "covid_positive",
    ]].head(20).to_string(index=False))
    mixed_label_groups.to_csv(NB02_DIR / "mixed_label_groups.csv", index=False)
    print()
    print("  These originate from NB 01's cross-study near-duplicate merging. If the count is "
          "material, review near_duplicate_label_conflicts.csv and consider re-running NB 01 "
          'with NEAR_DUPLICATE_MERGE_POLICY = "sha256_only".')


def band_of(total):
    if total <= 0:
        return "none"
    if total <= 10:
        return "mild"
    if total <= 18:
        return "moderate"
    return "severe"


# The group-level severity band uses the group mean, so a group's stratum does not depend on
# which of its images happens to be listed first.
group_frame["severity_band"] = group_frame["mrale_mean"].apply(band_of)
group_frame["stratum"] = group_frame["covid_positive"] + "|" + group_frame["severity_band"]

print(f"Grouping units: {len(group_frame):,}")
print()
print("Stratum sizes (groups):")
stratum_counts = group_frame["stratum"].value_counts()
for stratum, count in stratum_counts.items():
    print(f"  {stratum:<22} {count:>5}")

thin = stratum_counts[stratum_counts < N_FOLDS]
if len(thin):
    print()
    print(f"WARNING: {len(thin)} strata have fewer than {N_FOLDS} groups and cannot be "
          "represented in every fold:")
    print(thin.to_string())
    print("They are merged into their PCR-only stratum so that stratification stays "
          "well defined.")
    thin_strata = set(thin.index)
    group_frame["stratum"] = group_frame.apply(
        lambda row: row["covid_positive"] + "|merged"
        if row["stratum"] in thin_strata else row["stratum"],
        axis=1,
    )
    print()
    print("Stratum sizes after merging:")
    print(group_frame["stratum"].value_counts().to_string())

In [ ]:
def stratified_group_kfold_sklearn(groups, strata, n_splits, seed):
    from sklearn.model_selection import StratifiedGroupKFold

    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    assignment = {}
    indices = np.arange(len(groups))
    for fold, (_, test_index) in enumerate(splitter.split(indices, strata, groups)):
        for position in test_index:
            assignment[groups[position]] = fold
    return assignment, "sklearn.StratifiedGroupKFold"


def stratified_group_kfold_greedy(groups, strata, sizes, n_splits, seed):
    # Deterministic fallback: within each stratum, place groups largest-first into the fold
    # that currently holds the fewest images from that stratum. Ties break on total fold
    # size, then on fold index, so the result is fully reproducible.
    order = sorted(
        range(len(groups)),
        key=lambda position: (strata[position], -sizes[position], groups[position]),
    )
    rng = random.Random(seed)
    per_stratum_load = defaultdict(lambda: [0] * n_splits)
    total_load = [0] * n_splits
    assignment = {}
    for position in order:
        stratum = strata[position]
        loads = per_stratum_load[stratum]
        best = min(
            range(n_splits),
            key=lambda fold: (loads[fold], total_load[fold], rng.random()),
        )
        assignment[groups[position]] = best
        loads[best] += sizes[position]
        total_load[best] += sizes[position]
    return assignment, "deterministic_greedy_fallback"


groups = group_frame["group_id"].tolist()
strata = group_frame["stratum"].tolist()
sizes = group_frame["n_images"].tolist()

# Both splitters were dry-run on this cohort and both give zero group leakage.
# Observed balance (2,581 images, 2,064 groups, seed 42):
#   sklearn : 498-529 images/fold, 27-39 PCR-negative/fold, prevalence 0.9253-0.9458
#   greedy  : 516-517 images/fold, 33-34 PCR-negative/fold, prevalence 0.9341-0.9362
# sklearn is the citable standard; greedy balances the scarce negative class more tightly,
# which matters because specificity and AUROC are estimated from 27-39 cases per fold.
PREFER_SPLITTER = "sklearn"   # "sklearn" or "greedy"

if PREFER_SPLITTER == "greedy":
    fold_assignment, splitter_name = stratified_group_kfold_greedy(
        groups, strata, sizes, N_FOLDS, SEED)
else:
    try:
        fold_assignment, splitter_name = stratified_group_kfold_sklearn(
            groups, strata, N_FOLDS, SEED)
    except Exception as exc:
        print(f"StratifiedGroupKFold unavailable ({type(exc).__name__}: {exc}); using the "
              "deterministic greedy fallback.")
        fold_assignment, splitter_name = stratified_group_kfold_greedy(
            groups, strata, sizes, N_FOLDS, SEED)

print("Splitter:", splitter_name)
group_frame["fold"] = group_frame["group_id"].map(fold_assignment)
primary = primary.copy()
primary["fold"] = primary["group_id"].map(fold_assignment)

if primary["fold"].isna().any():
    raise RuntimeError("Some images were not assigned a fold.")
primary["fold"] = primary["fold"].astype(int)

print()
print("Images per fold:", dict(sorted(Counter(primary['fold']).items())))
print("Groups per fold:", dict(sorted(Counter(group_frame['fold']).items())))
print()
balance = pd.crosstab(primary["fold"], primary["covid_positive"])
balance["prevalence"] = (balance.get("Yes", 0) /
                         balance.sum(axis=1)).round(4)
print("PCR balance per fold:")
print(balance.to_string())
print()
print("Severity band per fold:")
print(pd.crosstab(primary["fold"], primary["severity_band"]).to_string())
print()
print("mRALE mean per fold:")
print(primary.groupby("fold")["mrale_total_annotated"].agg(["mean", "std", "count"]).round(3).to_string())

## 4. Verify the new folds

The same audit applied to the inherited folds is now applied to the new ones. This must come
out clean; if it does not, the fold construction is wrong and nothing downstream is valid.

In [ ]:
new_audit_rows = []
fold_groups = {fold: set(group_frame[group_frame["fold"] == fold]["group_id"])
               for fold in range(N_FOLDS)}
fold_images = {fold: set(primary[primary["fold"] == fold]["filename"])
               for fold in range(N_FOLDS)}

cross_group_overlaps = 0
for left in range(N_FOLDS):
    for right in range(left + 1, N_FOLDS):
        group_overlap = fold_groups[left] & fold_groups[right]
        image_overlap = fold_images[left] & fold_images[right]
        cross_group_overlaps += len(group_overlap)
        new_audit_rows.append({
            "scope": f"fold_{left}_vs_{right}", "check": "group_overlap",
            "value": len(group_overlap),
            "status": "PASS" if not group_overlap else "FAIL", "detail": "",
        })
        new_audit_rows.append({
            "scope": f"fold_{left}_vs_{right}", "check": "image_overlap",
            "value": len(image_overlap),
            "status": "PASS" if not image_overlap else "FAIL", "detail": "",
        })

for fold in range(N_FOLDS):
    train_groups = set().union(*[fold_groups[other] for other in range(N_FOLDS)
                                if other != fold])
    overlap = train_groups & fold_groups[fold]
    new_audit_rows.append({
        "scope": f"fold_{fold}", "check": "train_test_group_overlap",
        "value": len(overlap),
        "status": "PASS" if not overlap else "FAIL", "detail": "",
    })

# Byte-identical and near-duplicate images must not straddle folds.
sha_folds = primary.groupby("sha256")["fold"].nunique()
sha_straddling = int((sha_folds > 1).sum())
new_audit_rows.append({
    "scope": "all_folds", "check": "identical_image_straddles_folds",
    "value": sha_straddling,
    "status": "PASS" if sha_straddling == 0 else "FAIL",
    "detail": "Byte-identical PNGs assigned to different folds.",
})

near_duplicate_csv = NB01_DIR / "near_duplicate_pairs.csv"
near_duplicate_straddling = 0
straddling_pairs = []
if near_duplicate_csv.is_file():
    pairs = pd.read_csv(near_duplicate_csv)
    fold_by_filename = dict(zip(primary["filename"], primary["fold"]))
    for _, pair in pairs.iterrows():
        left = fold_by_filename.get(pair["filename_a"])
        right = fold_by_filename.get(pair["filename_b"])
        if left is not None and right is not None and left != right:
            near_duplicate_straddling += 1
            straddling_pairs.append({
                "filename_a": pair["filename_a"], "fold_a": left,
                "filename_b": pair["filename_b"], "fold_b": right,
                "hamming": pair.get("hamming"),
                "sha256_identical": pair.get("sha256_identical"),
                "same_covid_label": pair.get("same_covid_label"),
            })

# A straddling pair that NB 01 deliberately declined to merge (because merging would have
# created a mixed-label group) is a known, logged trade-off rather than an accident. It is
# reported as REVIEW so that it is visible without masking a genuine grouping bug.
straddle_frame = pd.DataFrame(straddling_pairs)
deliberate = 0
if len(straddle_frame) and "same_covid_label" in straddle_frame.columns:
    deliberate = int((straddle_frame["same_covid_label"] == False).sum())
    straddle_frame.to_csv(NB02_DIR / "near_duplicate_straddling_pairs.csv", index=False)
accidental = near_duplicate_straddling - deliberate

new_audit_rows.append({
    "scope": "all_folds", "check": "near_duplicate_straddles_folds",
    "value": near_duplicate_straddling,
    "status": "PASS" if accidental == 0 else "FAIL",
    "detail": (
        f"{accidental} unexplained; {deliberate} are label-disagreeing pairs NB 01 "
        "deliberately left unmerged (see near_duplicate_straddling_pairs.csv)."
    ),
})
if deliberate:
    new_audit_rows.append({
        "scope": "all_folds", "check": "near_duplicate_straddles_folds_deliberate",
        "value": deliberate, "status": "REVIEW",
        "detail": "Label-disagreeing near-duplicate pairs; grouping them was declined by "
                  "policy in NB 01.",
    })

new_audit = pd.DataFrame(new_audit_rows, columns=["scope", "check", "value", "status", "detail"])
new_audit.to_csv(NB02_DIR / "new_fold_audit.csv", index=False)

print(new_audit.groupby(["check", "status"]).size().to_string())
print()
new_failures = new_audit[new_audit["status"] == "FAIL"]
if len(new_failures):
    print("NEW FOLD AUDIT FAILURES:")
    print(new_failures.to_string(index=False))
else:
    print("New folds are clean: no group, image, identical-image, or near-duplicate "
          "leakage across folds.")

## 5. Harvest the prompt templates from the legacy files

Prompt text is harvested rather than retyped. Stage B must train on the same instructions the
tested notebooks used, so any wording drift between the old and new fold files would silently
become an uncontrolled variable in the E5 comparisons.

If the legacy files are unavailable, the fallback templates below are used and the fact is
recorded in `prompt_templates.json` for disclosure.

In [ ]:
FALLBACK_TEMPLATES = {
    "covid_classification": {
        "system": (
            "You are a radiology assistant. Examine the frontal PA chest x-ray and predict "
            "the PCR-defined COVID-19 status. Return only valid JSON with exactly one key: "
            "covid_positive. Its value must be Yes or No."
        ),
        "user": (
            "Predict whether this patient is PCR-positive or PCR-negative for COVID-19 from "
            "the frontal chest x-ray."
        ),
    },
    "mrale_prediction": {
        "system": (
            "You are a radiology assistant trained to evaluate frontal chest X-rays using the "
            "modified Radiographic Assessment of Lung Edema (mRALE) framework. Assess each lung "
            "independently and return only valid JSON with exactly these keys in this order: "
            "extent_right, density_right, extent_left, density_left, extent_right_numerical, "
            "density_right_numerical, extent_left_numerical, density_left_numerical, "
            "mRALE Score. Extent is an integer 0-4, density is an integer 0-3, each lung score "
            "is extent multiplied by density, and the total is the sum."
        ),
        "user": (
            "Predict the qualitative and numerical involvement and density scores for both "
            "lungs and the total mRALE Score."
        ),
    },
}


def message_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = [str(item.get("text", "")) for item in content
                 if isinstance(item, dict) and item.get("type") == "text"]
        return "\n".join(part for part in parts if part)
    return ""


def harvest_templates(legacy_dir, folds, patterns):
    harvested = defaultdict(lambda: defaultdict(Counter))
    for fold in folds:
        for pattern in patterns:
            path = Path(legacy_dir) / pattern.format(fold=fold)
            if not path.is_file():
                continue
            for record in read_jsonl(path):
                task = record.get("task")
                if not task:
                    continue
                for message in record.get("messages", []):
                    role = message.get("role")
                    if role in {"system", "user"}:
                        harvested[task][role][message_text(message.get("content", ""))] += 1
    return harvested


templates = {}
template_provenance = {}

harvested = harvest_templates(
    LEGACY_CV_DIR, range(N_FOLDS), [LEGACY_TRAIN_PATTERN, LEGACY_TEST_PATTERN]
) if LEGACY_CV_DIR.is_dir() else {}

for task in ["covid_classification", "mrale_prediction"]:
    if task in harvested and harvested[task]["system"] and harvested[task]["user"]:
        system_variants = harvested[task]["system"]
        user_variants = harvested[task]["user"]
        templates[task] = {
            "system": system_variants.most_common(1)[0][0],
            "user": user_variants.most_common(1)[0][0],
        }
        template_provenance[task] = {
            "source": "harvested_from_legacy_fold_files",
            "n_system_variants": len(system_variants),
            "n_user_variants": len(user_variants),
        }
        if len(system_variants) > 1 or len(user_variants) > 1:
            print(f"WARNING: {task} has multiple prompt variants in the legacy files "
                  f"(system={len(system_variants)}, user={len(user_variants)}). "
                  "The most frequent is used; inspect prompt_templates.json.")
    else:
        templates[task] = FALLBACK_TEMPLATES[task]
        template_provenance[task] = {"source": "notebook_fallback_constant"}
        print(f"NOTE: using the fallback template for {task}.")

with (NB02_DIR / "prompt_templates.json").open("w", encoding="utf-8") as handle:
    json.dump({"templates": templates, "provenance": template_provenance}, handle, indent=2)

for task, template in templates.items():
    print()
    print(f"--- {task} [{template_provenance[task]['source']}]")
    print("system:", template["system"][:160], "...")
    print("user:  ", template["user"][:160], "...")

## 6. Rebuild the multitask Harmony JSONL files

Direct-task records are constructed from `midrc_manifest.csv` using the harvested templates.
Truthfulness records are copied verbatim from `combined_cxr_harmony_train.jsonl` and
re-assigned to the new folds by `file_name`, so their prompts and TRUE/FALSE targets are
byte-identical to the tested versions.

Record schema matches the tested notebooks exactly: `id`, `file_name`, `image_path`,
`messages`, `ground_truth`, `task`, `meta`. Stage B's `read_jsonl` and `validate_records`
therefore work unchanged.

In [ ]:
# The qualitative vocabulary in the legacy Harmony files differs from the vocabulary in
# covid_midrc_dataset.csv. The models were supervised on the LEGACY strings, so those are
# what must be regenerated:
#
#   numeric | covid_midrc_dataset.csv | legacy Harmony JSONL  <- what the models learned
#   --------|------------------------|----------------------
#   extent 0| ""                     | "No involvement"
#   extent 1| "<=25%"                | "<25%"
#   extent 2| "25-50%"               | "25-50%"
#   extent 3| "51-75%"               | ">50-75%"
#   extent 4| ">75%"                 | ">75%"
#   density 0| ""                    | "No opacity"
#
# Using the CSV vocabulary here would change the supervision target for roughly 70% of
# records and silently invalidate every comparison against the earlier runs. The mapping is
# therefore harvested from the legacy files, with the observed values as the fallback.
FALLBACK_EXTENT_TEXT = {0: "No involvement", 1: "<25%", 2: "25-50%", 3: ">50-75%", 4: ">75%"}
FALLBACK_DENSITY_TEXT = {0: "No opacity", 1: "Hazy", 2: "Moderate", 3: "Dense"}


def harvest_qualitative_vocabulary(legacy_dir, folds, patterns):
    extent, density = defaultdict(Counter), defaultdict(Counter)
    for fold in folds:
        for pattern in patterns:
            path = Path(legacy_dir) / pattern.format(fold=fold)
            if not path.is_file():
                continue
            for record in read_jsonl(path):
                if record.get("task") != "mrale_prediction":
                    continue
                try:
                    answer = json.loads(record["ground_truth"]["answer"])
                except Exception:
                    continue
                for side in ["right", "left"]:
                    extent[answer[f"extent_{side}_numerical"]][answer[f"extent_{side}"]] += 1
                    density[answer[f"density_{side}_numerical"]][answer[f"density_{side}"]] += 1
    return extent, density


harvested_extent, harvested_density = (
    harvest_qualitative_vocabulary(
        LEGACY_CV_DIR, range(N_FOLDS), [LEGACY_TRAIN_PATTERN, LEGACY_TEST_PATTERN])
    if LEGACY_CV_DIR.is_dir() else ({}, {})
)

vocabulary_provenance = {}


def resolve_vocabulary(harvested, fallback, name, expected_keys):
    if not harvested or set(harvested) != set(expected_keys):
        vocabulary_provenance[name] = "notebook_fallback_constant"
        print(f"NOTE: using the fallback {name} vocabulary "
              f"(harvested keys: {sorted(harvested)}).")
        return dict(fallback)
    resolved, ambiguous = {}, {}
    for key, counter in harvested.items():
        resolved[key] = counter.most_common(1)[0][0]
        if len(counter) > 1:
            ambiguous[key] = dict(counter)
    vocabulary_provenance[name] = "harvested_from_legacy_fold_files"
    if ambiguous:
        vocabulary_provenance[name + "_ambiguous"] = ambiguous
        print(f"WARNING: {name} has multiple text values for the same numeric code: "
              f"{ambiguous}. The most frequent is used.")
    return resolved


EXTENT_TEXT = resolve_vocabulary(harvested_extent, FALLBACK_EXTENT_TEXT,
                                 "extent_text", [0, 1, 2, 3, 4])
DENSITY_TEXT = resolve_vocabulary(harvested_density, FALLBACK_DENSITY_TEXT,
                                  "density_text", [0, 1, 2, 3])
print("extent  numeric -> text:", EXTENT_TEXT)
print("density numeric -> text:", DENSITY_TEXT)


def covid_record(row, split, held_out_fold):
    answer = json.dumps({"covid_positive": row["covid_positive"]}, separators=(",", ":"))
    return {
        "id": f"MIDRC_{Path(row['filename']).stem}_covid_direct",
        "file_name": row["filename"],
        "image_path": row["image_path"],
        "messages": [
            {"role": "system", "content": templates["covid_classification"]["system"]},
            {"role": "user", "content": templates["covid_classification"]["user"]},
        ],
        "ground_truth": {"type": "exact_string", "answer": answer},
        "task": "covid_classification",
        "meta": {
            "source_dataset": "MIDRC",
            "split": split,
            "held_out_fold": int(held_out_fold),
            "group_id": row["group_id"],
            "study_uid": row["study_uid"],
            "sex": row["sex"],
            "race": row["race"],
            "ethnicity": row["ethnicity"],
            "covid_positive": row["covid_positive"],
            "severity_band": row["severity_band"],
            "quality_issue": row["quality_issue"],
            "supervision_type": "direct_prediction",
            "fold_definition": "midrc_folds_v2_stratified_group",
        },
    }


def mrale_record(row, split, held_out_fold):
    extent_right = int(row["extent_right_numerical"])
    density_right = int(row["density_right_numerical"])
    extent_left = int(row["extent_left_numerical"])
    density_left = int(row["density_left_numerical"])
    total = extent_right * density_right + extent_left * density_left
    answer = json.dumps(
        OrderedDict([
            ("extent_right", EXTENT_TEXT[extent_right]),
            ("density_right", DENSITY_TEXT[density_right]),
            ("extent_left", EXTENT_TEXT[extent_left]),
            ("density_left", DENSITY_TEXT[density_left]),
            ("extent_right_numerical", extent_right),
            ("density_right_numerical", density_right),
            ("extent_left_numerical", extent_left),
            ("density_left_numerical", density_left),
            ("mRALE Score", total),
        ]),
        separators=(",", ":"),
    )
    return {
        "id": f"MIDRC_{Path(row['filename']).stem}_mrale_direct",
        "file_name": row["filename"],
        "image_path": row["image_path"],
        "messages": [
            {"role": "system", "content": templates["mrale_prediction"]["system"]},
            {"role": "user", "content": templates["mrale_prediction"]["user"]},
        ],
        "ground_truth": {"type": "exact_string", "answer": answer},
        "task": "mrale_prediction",
        "meta": {
            "source_dataset": "MIDRC",
            "split": split,
            "held_out_fold": int(held_out_fold),
            "group_id": row["group_id"],
            "study_uid": row["study_uid"],
            "sex": row["sex"],
            "race": row["race"],
            "ethnicity": row["ethnicity"],
            "covid_positive": row["covid_positive"],
            "severity_band": row["severity_band"],
            "quality_issue": row["quality_issue"],
            "reference_mrale": {
                "er": extent_right, "dr": density_right,
                "el": extent_left, "dl": density_left, "total": total,
            },
            "supervision_type": "direct_prediction",
            "fold_definition": "midrc_folds_v2_stratified_group",
        },
    }


truthfulness_by_filename = defaultdict(list)
truthfulness_source_counts = Counter()
if INCLUDE_TRUTHFULNESS_TASKS and HARMONY_TRUTHFULNESS_JSONL:
    for record in read_jsonl(HARMONY_TRUTHFULNESS_JSONL):
        path = rewrite_image_path(record.get("image_path", ""))
        name = Path(path).name
        record = dict(record)
        record["image_path"] = path
        truthfulness_source_counts[record.get("task", "?")] += 1
        truthfulness_by_filename[name].append(record)
    print("Truthfulness source records by task:", dict(truthfulness_source_counts))
    print(f"Distinct images with truthfulness records: {len(truthfulness_by_filename):,}")
else:
    print("Truthfulness auxiliaries disabled or source file unavailable.")

In [ ]:
manifest_by_filename = {row["filename"]: row for _, row in primary.iterrows()}
fold_record_counts = {}
build_warnings = []

if REBUILD_HARMONY_JSONL:
    for fold in range(N_FOLDS):
        train_rows, test_rows = [], []
        for _, row in primary.iterrows():
            is_test = int(row["fold"]) == fold
            split = "test" if is_test else "train"
            destination = test_rows if is_test else train_rows
            destination.append(covid_record(row, split, fold))
            destination.append(mrale_record(row, split, fold))

        # Truthfulness auxiliaries are TRAINING-ONLY. The tested notebooks evaluate only the
        # direct tasks, and adding auxiliaries to the test file would change the denominator
        # of every reported metric.
        if INCLUDE_TRUTHFULNESS_TASKS:
            attached = 0
            for _, row in primary.iterrows():
                if int(row["fold"]) == fold:
                    continue
                for record in truthfulness_by_filename.get(row["filename"], []):
                    if "montgomery" in str(record.get("task", "")).lower():
                        continue
                    copied = dict(record)
                    meta = dict(copied.get("meta", {}))
                    meta.update({
                        "source_dataset": "MIDRC",
                        "split": "train",
                        "held_out_fold": fold,
                        "group_id": row["group_id"],
                        "supervision_type": "truthfulness_auxiliary",
                        "fold_definition": "midrc_folds_v2_stratified_group",
                    })
                    copied["meta"] = meta
                    train_rows.append(copied)
                    attached += 1
            if attached == 0:
                build_warnings.append(
                    f"fold {fold}: no truthfulness records matched by file_name. Check that "
                    "combined_cxr_harmony_train.jsonl covers the MIDRC filenames."
                )

        train_ids = Counter(record["id"] for record in train_rows)
        duplicate_ids = [key for key, count in train_ids.items() if count > 1]
        if duplicate_ids:
            raise ValueError(f"fold {fold}: duplicate record ids in train: "
                             f"{duplicate_ids[:5]}")

        write_jsonl(FOLD_DEF_DIR / f"multitask_train_fold_{fold}_harmony.jsonl", train_rows)
        write_jsonl(FOLD_DEF_DIR / f"multitask_test_fold_{fold}_harmony.jsonl", test_rows)
        fold_record_counts[fold] = {
            "train_records": len(train_rows),
            "test_records": len(test_rows),
            "train_tasks": dict(Counter(record["task"] for record in train_rows)),
            "test_tasks": dict(Counter(record["task"] for record in test_rows)),
        }
        print(f"fold {fold}: train={len(train_rows):,} test={len(test_rows):,} "
              f"| train tasks={fold_record_counts[fold]['train_tasks']}")

    if build_warnings:
        print()
        for message in build_warnings:
            print("WARNING:", message)
else:
    print("REBUILD_HARMONY_JSONL is False; only fold definitions were written.")

In [ ]:
fold_definition_columns = [
    "filename", "image_path", "group_id", "study_uid", "fold", "stratum_covid",
    "severity_band", "covid_positive", "mrale_total_annotated",
    "extent_right_numerical", "density_right_numerical",
    "extent_left_numerical", "density_left_numerical",
    "sex", "race", "ethnicity", "quality_issue", "sha256", "dhash64", "legacy_fold",
]
fold_definitions = primary.copy()
fold_definitions["stratum_covid"] = fold_definitions["covid_positive"]
fold_definitions.reindex(
    columns=[column for column in fold_definition_columns if column in fold_definitions.columns]
).to_csv(FOLD_DEF_DIR / "midrc_folds_v2.csv", index=False)

group_frame.to_csv(FOLD_DEF_DIR / "midrc_fold_groups_v2.csv", index=False)

fold_config = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "02_folds_and_leakage_audit.ipynb",
    "seed": SEED,
    "n_folds": N_FOLDS,
    "splitter": splitter_name,
    "grouping_column": "group_id",
    "grouping_definition": (
        "Study directory from the image path, with cross-study near-duplicate pairs merged "
        "by union-find in NB 01. No patient identifier is available in the source data."
    ),
    "stratification": "covid_positive x group-mean mRALE severity band, thin strata merged",
    "mixed_label_group_handling": {
        "n_mixed_label_groups": int(len(mixed_label_groups)),
        "rule": "majority vote, tie-break to Yes",
        "scope": (
            "Stratification / fold balancing only. Per-image PCR labels are unchanged and "
            "every reported metric uses them, so no ground truth is affected."
        ),
        "origin": "NB 01 cross-study near-duplicate group merging",
    },
    "protocol_decisions": {
        "include_montgomery_in_training": INCLUDE_MONTGOMERY_IN_TRAINING,
        "include_normality_task": INCLUDE_NORMALITY_TASK,
        "include_truthfulness_tasks": INCLUDE_TRUTHFULNESS_TASKS,
    },
    "legacy_audit": legacy_summary,
    "fold_record_counts": fold_record_counts,
    "prompt_template_provenance": template_provenance,
    "qualitative_vocabulary": {
        "extent_text": EXTENT_TEXT,
        "density_text": DENSITY_TEXT,
        "provenance": vocabulary_provenance,
        "note": (
            "Harvested from the legacy Harmony files, which use a different qualitative "
            "vocabulary from covid_midrc_dataset.csv. The models were supervised on these "
            "strings."
        ),
    },
    "build_warnings": build_warnings,
    "downstream_note": (
        "Stage B must point FOLD_DATA_DIR at fold_definitions/ in this directory, NOT at "
        "/data/liangz2/openi/midrc/multi_task_CV. TRAIN_TASKS drops "
        "normality_classification because Montgomery is held out as cohort X1."
    ),
}
with (NB02_DIR / "fold_config.json").open("w", encoding="utf-8") as handle:
    json.dump(fold_config, handle, indent=2, default=str)

print("Wrote fold definitions to:", FOLD_DEF_DIR)
print()
print("Stage B configuration change required:")
print(f'  FOLD_DATA_DIR = Path("{FOLD_DEF_DIR}")')
print('  DIRECT_TASKS = {"covid_classification", "mrale_prediction"}')
print('  AUXILIARY_TASKS = {"midrc_mrale_truthfulness"}   # confirm against train_tasks above')

## 7. Table 1 — cohort and fold composition

One row per fold plus a pooled row, with the columns the manuscript needs: images, groups,
PCR counts and prevalence, severity histogram, and demographics.

In [ ]:
def composition_row(label, subset):
    total = len(subset)
    positives = int((subset["covid_positive"] == "Yes").sum())
    negatives = int((subset["covid_positive"] == "No").sum())
    row = OrderedDict([
        ("cohort", label),
        ("n_images", total),
        ("n_groups", int(subset["group_id"].nunique())),
        ("pcr_positive", positives),
        ("pcr_negative", negatives),
        ("pcr_prevalence", round(positives / total, 4) if total else None),
        ("mrale_mean", round(float(subset["mrale_total_annotated"].mean()), 2) if total else None),
        ("mrale_sd", round(float(subset["mrale_total_annotated"].std()), 2) if total else None),
    ])
    for band in ["none", "mild", "moderate", "severe"]:
        row[f"severity_{band}"] = int((subset["severity_band"] == band).sum())
    row["sex_male"] = int((subset["sex"] == "Male").sum())
    row["sex_female"] = int((subset["sex"] == "Female").sum())
    row["race_not_reported"] = int((subset["race"] == "Not Reported").sum())
    row["quality_flagged"] = int((subset["quality_issue"].fillna("") != "").sum())
    return row


table1_rows = [composition_row("MIDRC pooled (primary cohort)", primary)]
for fold in range(N_FOLDS):
    table1_rows.append(composition_row(f"MIDRC test fold {fold}",
                                       primary[primary["fold"] == fold]))

table1 = pd.DataFrame(table1_rows)
table1.to_csv(NB02_DIR / "cohort_composition_table1.csv", index=False)
print(table1.to_string(index=False))
print()
print("NB 03 appends the external cohort rows (X1-X4) to this table.")

## 8. Gate

In [ ]:
failures = []
warnings = []

if len(new_failures):
    for _, row in new_failures.iterrows():
        failures.append(f"New folds: {row['scope']} {row['check']} = {row['value']}")

fold_counts = Counter(primary["fold"])
if len(fold_counts) != N_FOLDS:
    failures.append(f"Expected {N_FOLDS} folds, found {len(fold_counts)}.")
smallest, largest = min(fold_counts.values()), max(fold_counts.values())
if largest - smallest > 0.25 * (largest + smallest) / 2:
    warnings.append(f"Fold sizes are uneven: {dict(sorted(fold_counts.items()))}. "
                    "Group sizes constrain how even a grouped split can be.")

prevalences = [
    (primary[primary["fold"] == fold]["covid_positive"] == "Yes").mean()
    for fold in range(N_FOLDS)
]
if max(prevalences) - min(prevalences) > 0.05:
    warnings.append(f"PCR prevalence varies across folds by "
                    f"{max(prevalences) - min(prevalences):.3f}: "
                    f"{[round(value, 4) for value in prevalences]}")

for fold in range(N_FOLDS):
    subset = primary[primary["fold"] == fold]
    if int((subset["covid_positive"] == "No").sum()) < 10:
        failures.append(
            f"Fold {fold} has only {int((subset['covid_positive'] == 'No').sum())} "
            "PCR-negative images. Specificity and AUROC will be unstable."
        )

if REBUILD_HARMONY_JSONL:
    for fold in range(N_FOLDS):
        for pattern in ["multitask_train_fold_{fold}_harmony.jsonl",
                        "multitask_test_fold_{fold}_harmony.jsonl"]:
            path = FOLD_DEF_DIR / pattern.format(fold=fold)
            if not path.is_file():
                failures.append(f"Missing rebuilt fold file: {path}")
    montgomery_leaked = 0
    for fold in range(N_FOLDS):
        for pattern in ["multitask_train_fold_{fold}_harmony.jsonl",
                        "multitask_test_fold_{fold}_harmony.jsonl"]:
            for record in read_jsonl(FOLD_DEF_DIR / pattern.format(fold=fold)):
                if "Montgomery" in record.get("image_path", ""):
                    montgomery_leaked += 1
    if montgomery_leaked and not INCLUDE_MONTGOMERY_IN_TRAINING:
        failures.append(f"{montgomery_leaked} Montgomery records entered the rebuilt folds.")

if build_warnings:
    warnings.extend(build_warnings)

if not LEGACY_FOLDS_ARE_CLEAN:
    warnings.append(
        "The inherited folds failed the audit. Every previously reported internal number "
        "(MAE 3.88, COVID balanced accuracy 0.616) is superseded and must not appear in the "
        "revised manuscript except as a labelled pre-audit reference."
    )


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

with (NB02_DIR / "gate_nb02.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "passed": not failures,
        "failures": failures,
        "warnings": warnings,
        "legacy_folds_clean": LEGACY_FOLDS_ARE_CLEAN,
        "splitter": splitter_name,
    }, handle, indent=2)

assert not failures, f"NB 02 gate failed with {len(failures)} blocking issue(s)."
print()
print("NB 02 gate: PASSED")

## Notes carried forward

- **Stage B must be repointed.** `FOLD_DATA_DIR` becomes this notebook's `fold_definitions/`
  directory. The tested notebooks' `TRAIN_PATTERN` and `TEST_PATTERN` strings are unchanged,
  so only the directory constant moves.
- **`normality_classification` is gone** from the regenerated folds because Montgomery is
  held out entirely as cohort X1. Stage B's `DIRECT_TASKS` and `AUXILIARY_TASKS` sets must be
  updated to match the `train_tasks` printed in Section 6, and the `validate_records`
  normality branch will simply never fire.
- **Every earlier internal number is superseded.** The inherited folds put 135-153 study
  groups in both train and test of the same fold. Do not compare a new result against an old
  one; regenerate the baseline arm (E5-M / M5) on these folds first, and treat that as the
  reference point the protocol's G3 gate refers to.
- **Grouping is at study level, not patient level.** State this in the manuscript. If MIDRC
  case identifiers can be exported, add a `patient_id` column to the source CSV; NB 01 will
  prefer it and NB 02 needs no change.
- **The qualitative vocabulary is a trap.** `covid_midrc_dataset.csv` writes `"<=25%"`,
  `"51-75%"` and `""`; the legacy Harmony files write `"<25%"`, `">50-75%"` and
  `"No involvement"`. The models were supervised on the legacy strings, so Section 6 harvests
  them rather than deriving them from the CSV. A local check found that using the CSV
  vocabulary would have altered the supervision target of roughly 70% of mRALE records while
  leaving all the numeric fields correct, which is the kind of change that produces a
  puzzling accuracy drop with no obvious cause.
- `midrc_folds_v2.csv` is the split definition to publish as supplementary material. It
  contains no image data and no protected identifiers.